# Territory overlays

**Output family:** TERRITORY OVERLAYS

Builds the service territory overlay: the electric network outlines, the gas service areas, and the ORU outlines the map draws beneath the tracts.

**What it produces**

- `Data/service_territories.geojson`
- `Data/out/tract_geometry_pure-2010.json  (the same command produces this too)`

## Where this sits in the execution order

The guide filenames are numbered by output family, **not** by sequence, so
the order they run in is not 1, 2, 3. This is the order:

| | Guide | Command | Produces |
|---|---|---|---|
| Step 1 **<- you are here** | `docs/02-geometry-and-territories.html` | `python scripts/update_map_data.py --vintage 2010` | tract shapes AND the territory overlay: one command, two outputs |
| Step 2 | `docs/01-nyserda-indicator-dataset.html` | `python scripts/convert_nyserda_raw.py --version 1.0 --geoid-vintage 2010 --raw-date 2023-03-27` | the DAC indicator dataset |
| Step 3 | `docs/03-electric-and-gas-figures.html` | `python scripts/build_coned_dataset.py --vintage 2010` | the electric and gas figures |


> **One command, two outputs.** `update_map_data.py` is the single producer of both the tract shapes and the territory overlay. It builds them together, from one set of shapefiles, and stamps both with the same fingerprint so they cannot drift apart. That is why this notebook and the other one for step 1 run the same command: whichever you run, you get both files.

> **`--refresh-territories` is not an overlay-only rebuild.** Verified, not assumed: the run always continues on to rebuild the dataset as well, so with a dataset already in `Data/out/` the flag needs `--force` too and both files are rewritten. There is no supported way to rebuild the overlay alone, and that is deliberate -- the two outputs are stamped as a pair.

## How to use this notebook

Run the cells in order. This notebook is an **orchestrator**: it calls the
package's own scripts and reimplements nothing, so what it produces is
byte-identical to running the same commands in a terminal.

Nothing here contacts the dashboard, Dataverse, or any Con Edison system.
Uploading is a separate manual step, described in the guide.


## 1. Find the package


In [ ]:
# Find the unpacked package. Nothing here writes anything.
import hashlib, os, subprocess, sys, json, glob

# If you unpacked somewhere this does not find, set it by hand:
#   PKG = '/content/coned-dac-dashboard-data-tools'
PKG = None

def _looks_like_pkg(d):
    return (os.path.isfile(os.path.join(d, 'MANIFEST.txt'))
            and os.path.isdir(os.path.join(d, 'scripts'))
            and os.path.isdir(os.path.join(d, 'Data')))

if PKG is None:
    here = os.getcwd()
    candidates = [here, os.path.dirname(here)]
    candidates += sorted(glob.glob('/content/**/coned-dac-dashboard-data-tools',
                                   recursive=True))
    candidates += sorted(glob.glob(os.path.join(here, '**',
                         'coned-dac-dashboard-data-tools'), recursive=True))
    for c in candidates:
        if c and _looks_like_pkg(c):
            PKG = c
            break

if PKG is None:
    raise SystemExit('Could not find the package root. Unpack the zip, then set '
                     'PKG above to the folder holding MANIFEST.txt, scripts/ and Data/.')

print('package root :', PKG)
print('contents     :', ', '.join(sorted(os.listdir(PKG))))


## 2. Integrity: is this package intact?


In [ ]:
# INTEGRITY. Two halves, and the second is the one that proves something.
#
# 1. The zip's own size and sha256, printed for you to compare against the
#    figures in the handoff note. A notebook INSIDE the zip cannot contain
#    the zip's own digest, so this prints rather than asserts.
# 2. Every unpacked file against the full sha256 in MANIFEST.txt. This is
#    the real check, and it is only possible because MANIFEST.txt carries
#    complete 64-character digests.

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

zips = sorted(glob.glob('/content/**/coned-dac-dashboard-data-tools*.zip',
                        recursive=True))
zips += sorted(glob.glob(os.path.join(os.path.dirname(PKG),
                         'coned-dac-dashboard-data-tools*.zip')))
if zips:
    z = zips[0]
    print('zip          :', z)
    print('  bytes      :', os.path.getsize(z))
    print('  sha256     :', sha256_of(z))
else:
    print('zip          : not found (fine: it may have been deleted after unpacking)')

# MANIFEST.txt rows look like:
#   <path>  <bytes>
#       sha256  <64 hex>
rows, path_now = [], None
with open(os.path.join(PKG, 'MANIFEST.txt'), encoding='utf-8') as fh:
    for line in fh:
        s = line.rstrip('\n')
        t = s.strip()
        if t.startswith('sha256 ') and path_now:
            rows.append((path_now[0], path_now[1], t.split(None, 1)[1].strip()))
            path_now = None
        elif s.startswith(' ') or not t or t.startswith('-') or t.startswith('='):
            continue
        else:
            parts = t.rsplit(None, 1)
            if len(parts) == 2 and parts[1].isdigit():
                path_now = (parts[0].strip(), int(parts[1]))

bad, checked = [], 0
for rel, size, digest in rows:
    full = os.path.join(PKG, rel)
    if not os.path.exists(full):
        bad.append('%s: listed in MANIFEST, absent from the package' % rel)
        continue
    if len(digest) != 64:
        bad.append('%s: MANIFEST digest is %d characters, not 64' % (rel, len(digest)))
        continue
    actual_size, actual = os.path.getsize(full), sha256_of(full)
    if actual_size != size:
        bad.append('%s: %d bytes on disk, MANIFEST says %d' % (rel, actual_size, size))
    if actual != digest:
        bad.append('%s: sha256 mismatch' % rel)
    checked += 1

print()
print('MANIFEST rows parsed   :', len(rows))
print('files verified         :', checked)
if bad:
    print('PROBLEMS               :', len(bad))
    for b in bad:
        print('   ', b)
    raise SystemExit('the package does not match its own MANIFEST; stopping.')
print('every file matches MANIFEST.txt on both size and full sha256.')


## 3. Install the dependencies


In [ ]:
# Dependencies, from the package's own requirements.txt. Nothing pinned
# here by hand: the file in the package is the source of truth.
req = os.path.join(PKG, 'scripts', 'requirements.txt')
print(open(req, encoding='utf-8').read())
p = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req],
                   capture_output=True, text=True)
print(p.stdout[-2000:])
print(p.stderr[-2000:])
if p.returncode != 0:
    raise SystemExit('pip install failed with %d' % p.returncode)
print('dependencies installed.')


## 4. The run helper


In [ ]:
# The run helper. Every command below goes through this, so the command
# actually issued is visible and a non-zero exit stops the notebook instead
# of scrolling past.
#
# cwd is the PACKAGE ROOT, which is where the guides say to run from: the
# scripts resolve Data/ from their own location, and the paths they print are
# relative to the root.
def run(args, expect=0):
    print('$ python ' + ' '.join(args))
    print('-' * 70)
    p = subprocess.run([sys.executable] + args, cwd=PKG,
                       capture_output=True, text=True)
    sys.stdout.write(p.stdout)
    if p.stderr.strip():
        print('--- stderr ---')
        sys.stdout.write(p.stderr)
    print('-' * 70)
    print('exit code:', p.returncode)
    if expect is not None and p.returncode != expect:
        raise SystemExit('expected exit %s, got %d' % (expect, p.returncode))
    return p


## 5. Dry run: preflight only

The preflight names what it will do with the overlay: `WILL BUILD` when
it is absent, `WILL REBUILD` when its stamp disagrees with the
shapefiles, `PRESENT` when it already matches.


In [ ]:
run(['scripts/update_map_data.py', '--vintage', '2010', '--dry-run'])


## 6. The real run

The same single command as step 1, because one command produces both
outputs. The overlay lands in `Data/`, **not** in `Data/out/`.


In [ ]:
run(['scripts/update_map_data.py', '--vintage', '2010'])


## Verify the outputs

Sizes and digests are measured from the files the run just wrote.


In [ ]:
# VERIFICATION. Measure what the run produced, and compare it against
# values that were measured from a real build. A value of None means no
# measurement exists yet: the cell records what it found and says so rather
# than comparing against a number nobody measured.
EXPECTED = [
    ('Data/service_territories.geojson', 239162, 'a45ae7f05d4aac95cf37d6f77aa4a5d536b4da6dcf08de155a318a19e10ea4c7'),
    ('Data/out/tract_geometry_pure-2010.json', 1587328, '6e9f09f7414f59ed4381b59d0d66bb6baa24f5c162182934b4f9c7d756413f49'),
]

problems = []
for rel, exp_size, exp_sha in EXPECTED:
    full = os.path.join(PKG, rel)
    print(rel)
    if not os.path.exists(full):
        print('    MISSING: the run did not produce this file')
        problems.append('%s is missing' % rel)
        continue
    size, digest = os.path.getsize(full), sha256_of(full)
    print('    bytes  :', size,
          '' if exp_size is None else ('(expected %d)' % exp_size))
    print('    sha256 :', digest)
    if exp_sha is None:
        print('    expected sha256: not measured; recorded, not compared')
    else:
        print('    expected       :', exp_sha)
    if exp_size is not None and size != exp_size:
        problems.append('%s: %d bytes, expected %d' % (rel, size, exp_size))
    if exp_sha is not None and digest != exp_sha:
        problems.append('%s: sha256 differs from the expected build' % rel)

# Layer counts, read out of the overlay itself.
terr = json.load(open(os.path.join(PKG, 'Data/service_territories.geojson'), encoding='utf-8'))
feats = terr['features']
per = {}
for f_ in feats:
    k = f_['properties'].get('layer', '?')
    per[k] = per.get(k, 0) + 1
print()
print('kind     :', terr.get('kind'))
print('features :', len(feats))
for k in sorted(per):
    print('    %-10s %d' % (k, per[k]))
print('sourceFingerprint:', str(terr.get('sourceFingerprint'))[:16])
if len(per) < 3:
    problems.append('expected three layers, found %d' % len(per))

print()
if problems:
    for p_ in problems:
        print('PROBLEM:', p_)
    raise SystemExit('verification failed.')
print('verification passed.')


## What happens next

Nothing in this notebook uploads anything. Follow the guide's upload
section to publish these files through the dashboard's Map Layers card.
